# 🧬 Adverse Event NLP Pipeline
## Clinical Insights Extraction from Pharmaceutical Literature

**Author:** Matt Derya | Data Scientist | 15+ years Pharma | mattderya.com

## 🎯 Project Background

During my work at **Mentor R&D**, I developed NLP pipelines to automatically extract clinical insights from **scientific literature and adverse event reports** — a critical capability for Oncology and Immunology drug development programs.

**The Problem:**
- Manual abstract screening: weeks per drug review cycle
- Human error in classifying millions of clinical sentences
- Delayed adverse event signal detection affecting patient safety

**The Solution:**
- Automated NLP pipeline using **Transformers / HuggingFace**
- Sentence-level classification of medical abstracts
- Rapid identification of Results & Adverse Event signals

## ⚠️ Important Note — Replica Notebook

> This notebook is a **simplified, reproducible replica** of the actual production NLP pipeline developed at **Mentor R&D** for pharmaceutical literature analysis.

**Why a replica?**
- 🔒 **Data Privacy & HIPAA Compliance:** The original pipeline processed proprietary adverse event reports and internal clinical documents that cannot be shared publicly.
- 📋 The **PubMed 200k RCT** dataset is used here as a publicly available proxy.

| | This Notebook | Production Pipeline |
|---|---|---|
| **Dataset** | Public PubMed 200k RCT | Proprietary adverse event reports |
| **Model** | DistilBERT | BioBERT fine-tuned |
| **Scale** | ~50k samples | Millions of documents |

**Production Results (Mentor R&D):**
- ⚡ Abstract screening time reduced from **weeks → hours**
- 🎯 Enabled faster adverse event signal detection for **Oncology/Immunology** programs
- 🏥 Full **HIPAA compliance** maintained throughout

In [ ]:
# ============================================================
# ADVERSE EVENT NLP PIPELINE
# Author: Matt Derya | Data Scientist | mattderya.com
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

print(f'TensorFlow version: {tf.__version__}')
print('✅ Libraries loaded successfully')
print('🧬 Adverse Event NLP Pipeline Ready')

## 📊 1. Data Exploration & Clinical Context

The **PubMed 200k RCT** dataset contains sentences from randomized controlled trial abstracts, each labeled with its rhetorical role:

- **BACKGROUND** — Study context and rationale
- **OBJECTIVE** — Research goals
- **METHODS** — Study design and procedures
- **RESULTS** — Findings and outcomes ← *Key for adverse event detection*
- **CONCLUSIONS** — Clinical implications

In [ ]:
# ============================================================
# DATA LOADING & EXPLORATION
# ============================================================

train_df = pd.read_csv('/kaggle/input/pubmed-rct200k/train.csv')
val_df = pd.read_csv('/kaggle/input/pubmed-rct200k/dev.csv')
test_df = pd.read_csv('/kaggle/input/pubmed-rct200k/test.csv')

print(f'Training samples: {len(train_df):,}')
print(f'Validation samples: {len(val_df):,}')
print(f'Test samples: {len(test_df):,}')
print(f'\nLabel distribution:')
print(train_df['label'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PubMed 200k RCT — Clinical Sentence Distribution', fontsize=14, fontweight='bold')

colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6']
counts = train_df['label'].value_counts()

axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Label Distribution')

axes[1].bar(counts.index, counts.values, color=colors, edgecolor='black')
axes[1].set_title('Sample Count by Label')
axes[1].set_ylabel('Number of Samples')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Data exploration complete')

## 🔬 2. Text Preprocessing & Analysis

Pharmaceutical NLP requires careful text preprocessing to handle:
- Medical abbreviations and terminology
- Complex sentence structures in clinical writing
- Sentence length variations across abstract sections

In [ ]:
# ============================================================
# TEXT PREPROCESSING & ANALYSIS
# ============================================================

train_df['text_length'] = train_df['text'].str.len()
train_df['word_count'] = train_df['text'].str.split().str.len()

print('📊 Text Statistics:')
print(train_df.groupby('label')[['text_length', 'word_count']].mean().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Clinical Sentence Length Analysis by Category', fontsize=14, fontweight='bold')

for label in train_df['label'].unique():
    subset = train_df[train_df['label'] == label]
    axes[0].hist(subset['word_count'], alpha=0.6, label=label, bins=30)

axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Word Count Distribution')
axes[0].legend()

sns.boxplot(data=train_df, x='label', y='word_count', ax=axes[1], palette='Set2')
axes[1].set_title('Word Count by Label')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('text_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Text analysis complete')

## 🤖 3. DistilBERT Fine-tuning for Clinical Classification

**DistilBERT** is used as a computationally efficient proxy for **BioBERT**, which was used in the actual production pipeline.

In [ ]:
# ============================================================
# TOKENIZATION & MODEL PREPARATION
# ============================================================

SAMPLE_SIZE = 10000
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 3
SEED = 42

train_sample = train_df.sample(SAMPLE_SIZE, random_state=SEED)
val_sample = val_df.sample(2000, random_state=SEED)

le = LabelEncoder()
train_labels = le.fit_transform(train_sample['label'])
val_labels = le.transform(val_sample['label'])
num_classes = len(le.classes_)

print(f'Classes: {le.classes_}')
print(f'Training samples: {len(train_sample):,}')

print('\nLoading DistilBERT tokenizer...')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts, max_length=MAX_LENGTH):
    return tokenizer(
        list(texts),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )

print('Tokenizing data...')
train_encodings = tokenize(train_sample['text'])
val_encodings = tokenize(val_sample['text'])

train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings), train_labels
)).shuffle(1000).batch(BATCH_SIZE)

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings), val_labels
)).batch(BATCH_SIZE)

print('✅ Data pipeline ready')

In [ ]:
# ============================================================
# MODEL TRAINING
# ============================================================

print('Loading DistilBERT model...')
model = TFDistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_classes
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

print(f"\n{'='*50}")
print('🧬 Training: DistilBERT Clinical Classifier')
print(f"{'='*50}")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    verbose=1
)

best_val_acc = max(history.history['val_accuracy'])
print(f'\n✅ Best Validation Accuracy: {best_val_acc*100:.2f}%')

## 📈 4. Results Analysis & Adverse Event Insights

In [ ]:
# ============================================================
# RESULTS ANALYSIS
# ============================================================

print('Generating predictions...')
val_preds = model.predict(val_dataset)
val_pred_labels = np.argmax(val_preds.logits, axis=1)

print('\n📊 CLASSIFICATION REPORT:')
print(classification_report(val_labels, val_pred_labels, target_names=le.classes_))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Adverse Event NLP Pipeline — Results Analysis', fontsize=14, fontweight='bold')

cm = confusion_matrix(val_labels, val_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

axes[1].plot(history.history['accuracy'], label='Train Acc', marker='o')
axes[1].plot(history.history['val_accuracy'], label='Val Acc', marker='s')
axes[1].set_title('Training History')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"""
╔══════════════════════════════════════════════════╗
║     ADVERSE EVENT NLP PIPELINE — FINAL RESULTS   ║
╠══════════════════════════════════════════════════╣
║  Model         : DistilBERT                      ║
║  Val Accuracy  : {best_val_acc*100:.2f}%                         ║
║  Classes       : 5 (Background, Objective,       ║
║                  Methods, Results, Conclusions)  ║
║  Clinical Use  : Adverse Event Signal Detection  ║
╚══════════════════════════════════════════════════╝
""")
print('✅ Pipeline complete!')